
# NoShowGuard Project
## Part 3: Machine Learning — No-Show Prediction

**Objective:** Predict appointment no-show risk using machine learning models.

**Input Dataset:** appointments_clean.csv

**Tasks Covered:**
- Task 3.1: Features, Train-Test Split and Baseline
- Task 3.2: Logistic Regression
- Task 3.3: Decision Tree and Overfitting
- Task 3.4: Random Forest and Model Comparison
- Task 3.5: Hyperparameter Tuning, Model Selection and Threshold


In [2]:
# Import pandas for data handling
import pandas as pd

# Load the cleaned dataset generated in Part 1
df_ml = pd.read_csv("appointments_clean.csv")

# Display dataset dimensions
print("Dataset Shape:", df_ml.shape)

# Display first 5 rows
df_ml.head()

Dataset Shape: (990, 16)


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850.0,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350.0,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850.0,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750.0,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700.0,Phone,Consultation,No,0,No-Show,40,1


In [3]:
# Select the required input features
X = df_ml[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# Convert reminder_sent from Yes/No into 1/0
X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

# Apply one-hot encoding to categorical features
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

# Define the target variable
y = df_ml["no_show"]

# Display feature and target information
print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nFeature Columns:")
print(X.columns.tolist())

print("\nTarget Distribution:")
print(y.value_counts())

Feature Shape: (990, 11)
Target Shape: (990,)

Feature Columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Target Distribution:
no_show
0    772
1    218
Name: count, dtype: int64


In [5]:
import sys
print(sys.executable)

c:\Users\Drishan\NoShowGuard\venv\Scripts\python.exe


In [ ]:
# Import train_test_split-- task 3.1
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Calculate baseline accuracy
# Baseline always predicts Attended (0)
baseline_accuracy = (y_test == 0).mean()

# Display split details
print("Training Set Size:", X_train.shape)
print("Testing Set Size:", X_test.shape)

print("\nNo-shows in Test Set:", y_test.sum())

print("Baseline Accuracy:", round(baseline_accuracy, 4))

Training Set Size: (792, 11)
Testing Set Size: (198, 11)

No-shows in Test Set: 49
Baseline Accuracy: 0.7525


In [7]:
# Import required libraries-- task 3.2
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Initialize StandardScaler
scaler = StandardScaler()

# Fit scaler on training data only
X_train_s = scaler.fit_transform(X_train)

# Transform test data using the same scaler
X_test_s = scaler.transform(X_test)

print("Training data scaled:", X_train_s.shape)
print("Testing data scaled:", X_test_s.shape)

Training data scaled: (792, 11)
Testing data scaled: (198, 11)


In [8]:
# Initialize Logistic Regression model
model = LogisticRegression(max_iter=1000)

# Train the model using scaled training data
model.fit(X_train_s, y_train)

# Generate predictions on test data
y_pred = model.predict(X_test_s)

# Generate no-show probabilities
y_prob = model.predict_proba(X_test_s)[:, 1]

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [9]:
# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)

conf_matrix = confusion_matrix(y_test, y_pred)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_prob)

# Display results
print("Logistic Regression - Test Performance")
print("--------------------------------------")

print("Accuracy:", round(accuracy, 4))

print("\nConfusion Matrix:")
print(conf_matrix)

print("\nPrecision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))
print("ROC-AUC:", round(roc_auc, 4))

Logistic Regression - Test Performance
--------------------------------------
Accuracy: 0.7879

Confusion Matrix:
[[139  10]
 [ 32  17]]

Precision: 0.6296
Recall: 0.3469
F1 Score: 0.4474
ROC-AUC: 0.7891


In [10]:
import numpy as np

# Get Logistic Regression coefficients
coefficients = model.coef_[0]

# Create a DataFrame for feature coefficients
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": coefficients,
    "Odds_Ratio": np.exp(coefficients)
})

# Display coefficients
print("Logistic Regression Feature Coefficients")
print("-----------------------------------------")

print(coef_df.to_string(index=False))

# Calculate reminder effect for actual No-to-Yes change
reminder_index = list(X.columns).index("reminder_sent")

reminder_coef = coefficients[reminder_index]
reminder_scale = scaler.scale_[reminder_index]

actual_reminder_odds = np.exp(reminder_coef / reminder_scale)

print("\nReminder Sent Analysis")
print("Coefficient:", round(reminder_coef, 4))
print("Odds Ratio per Standard Deviation:", round(np.exp(reminder_coef), 4))
print("Odds Ratio for No to Yes:", round(actual_reminder_odds, 4))

Logistic Regression Feature Coefficients
-----------------------------------------
                Feature  Coefficient  Odds_Ratio
                    age    -0.253872    0.775791
            distance_km     0.253433    1.288441
                fee_inr     0.452784    1.572685
              lead_days     0.614443    1.848627
         prior_no_shows     0.649593    1.914762
          reminder_sent    -0.580643    0.559539
    booking_channel_App    -0.046799    0.954279
  booking_channel_Phone     0.215886    1.240961
booking_channel_Walk-In    -0.186374    0.829963
visit_type_Consultation     0.015649    1.015772
   visit_type_Follow-up    -0.015649    0.984473

Reminder Sent Analysis
Coefficient: -0.5806
Odds Ratio per Standard Deviation: 0.5595
Odds Ratio for No to Yes: 0.3094


**Logistic Regression — Interpretation**

The reminder_sent feature has a negative coefficient of -0.5806. Its odds ratio per standard deviation is 0.5595. However, for the actual change from No (0) to Yes (1), the odds ratio is 0.3094.

In this model, holding the other features constant, appointments with a reminder are associated with approximately 69% lower odds of a no-show compared with appointments without a reminder. This is an association in observational data and does not prove that reminders cause fewer no-shows.

The lead_days and prior_no_shows features have positive coefficients, indicating that higher values are associated with increased no-show odds in this model.

In [11]:
# Import Decision Tree Classifier-- task 3.3
from sklearn.tree import DecisionTreeClassifier

# Create an unrestricted Decision Tree
dt_full = DecisionTreeClassifier(random_state=42)

# Train the full Decision Tree
dt_full.fit(X_train, y_train)

# Calculate training and testing accuracy
full_train_accuracy = dt_full.score(X_train, y_train)
full_test_accuracy = dt_full.score(X_test, y_test)

# Create a Decision Tree with maximum depth of 3
dt_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# Train the depth-3 Decision Tree
dt_depth3.fit(X_train, y_train)

# Calculate training and testing accuracy
depth3_train_accuracy = dt_depth3.score(X_train, y_train)
depth3_test_accuracy = dt_depth3.score(X_test, y_test)

# Display results
print("Full Decision Tree")
print("Training Accuracy:", round(full_train_accuracy, 4))
print("Testing Accuracy:", round(full_test_accuracy, 4))
print("Tree Depth:", dt_full.get_depth())

print("\nDepth-3 Decision Tree")
print("Training Accuracy:", round(depth3_train_accuracy, 4))
print("Testing Accuracy:", round(depth3_test_accuracy, 4))

Full Decision Tree
Training Accuracy: 1.0
Testing Accuracy: 0.6717
Tree Depth: 15

Depth-3 Decision Tree
Training Accuracy: 0.822
Testing Accuracy: 0.7778


**Decision Tree — Overfitting Analysis**

The unrestricted Decision Tree achieved 100% training accuracy but only 67.17% testing accuracy. Its depth of 15 indicates that it has likely memorised patterns and noise in the training data, resulting in overfitting.

The depth-3 Decision Tree achieved 82.20% training accuracy and 77.78% testing accuracy. Limiting the tree depth reduced overfitting and improved its ability to generalise to unseen data.

In [12]:
# Import Random Forest Classifier-- task 3.4
from sklearn.ensemble import RandomForestClassifier

# Initialize Random Forest model
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the Random Forest model
rf.fit(X_train, y_train)

# Generate predictions on test data
rf_pred = rf.predict(X_test)

# Generate no-show probabilities
rf_prob = rf.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_roc_auc = roc_auc_score(y_test, rf_prob)

# Display performance
print("Random Forest Performance")
print("-------------------------")
print("Training Accuracy:", round(rf.score(X_train, y_train), 4))
print("Testing Accuracy:", round(rf_accuracy, 4))
print("Recall:", round(rf_recall, 4))
print("ROC-AUC:", round(rf_roc_auc, 4))

Random Forest Performance
-------------------------
Training Accuracy: 1.0
Testing Accuracy: 0.7525
Recall: 0.3061
ROC-AUC: 0.7041


In [13]:
# Create a DataFrame for feature importance
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

# Sort features by importance in descending order
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

# Display top 3 important features
print("Top 3 Important Features")
print("------------------------")
print(importance_df.head(3).to_string(index=False))

Top 3 Important Features
------------------------
    Feature  Importance
  lead_days    0.218094
        age    0.196555
distance_km    0.195410


In [14]:
# Create a comparison table for all three models as specified
comparison_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy,
        depth3_test_accuracy,
        rf_accuracy
    ],
    "Recall": [
        recall,
        recall_score(y_test, dt_depth3.predict(X_test)),
        rf_recall
    ],
    "ROC-AUC": [
        roc_auc,
        roc_auc_score(
            y_test,
            dt_depth3.predict_proba(X_test)[:, 1]
        ),
        rf_roc_auc
    ]
})

# Display comparison table
print("Model Performance Comparison")
print("----------------------------")

print(comparison_df.round(4).to_string(index=False))

Model Performance Comparison
----------------------------
                  Model  Accuracy  Recall  ROC-AUC
    Logistic Regression    0.7879  0.3469   0.7891
Decision Tree (Depth 3)    0.7778  0.2245   0.6997
          Random Forest    0.7525  0.3061   0.7041


**Model Comparison — Initial Observation**

Logistic Regression achieved the highest test ROC-AUC among the three models, indicating better ability to distinguish between no-show and attended appointments. The depth-3 Decision Tree provided reasonable accuracy with less overfitting, while the Random Forest achieved moderate performance. However, the formal model selection will be based on cross-validation using training data only in Task 3.5.

In [15]:
# Import GridSearchCV-- task 3.5
from sklearn.model_selection import GridSearchCV

# Initialize Random Forest
rf_grid_model = RandomForestClassifier(random_state=42)

# Define hyperparameter combinations
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=rf_grid_model,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc"
)

# Fit GridSearchCV on training data only
grid_search.fit(X_train, y_train)

# Display best parameters and CV score
print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation ROC-AUC:", round(grid_search.best_score_, 4))

Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best Cross-Validation ROC-AUC: 0.768


In [16]:
# Import cross-validation function
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# Create Logistic Regression Pipeline
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Calculate 5-fold CV ROC-AUC for Logistic Regression
lr_cv_auc = cross_val_score(
    lr_pipe, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

# Calculate CV ROC-AUC for Depth-3 Decision Tree
dt_cv_auc = cross_val_score(
    dt_depth3, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

# Calculate CV ROC-AUC for Random Forest
rf_cv_auc = cross_val_score(
    rf, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

# Get tuned Random Forest CV ROC-AUC
tuned_rf_cv_auc = grid_search.best_score_

# Create model selection comparison
cv_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "CV ROC-AUC": [
        lr_cv_auc,
        dt_cv_auc,
        rf_cv_auc,
        tuned_rf_cv_auc
    ]
})

# Display results
print("5-Fold Cross-Validation Model Selection")
print("---------------------------------------")
print(cv_comparison.round(4).to_string(index=False))

# Select the model with the highest CV ROC-AUC
best_model_name = cv_comparison.loc[
    cv_comparison["CV ROC-AUC"].idxmax(), "Model"
]

print("\nSelected Model:", best_model_name)

5-Fold Cross-Validation Model Selection
---------------------------------------
                  Model  CV ROC-AUC
    Logistic Regression      0.7737
Decision Tree (Depth 3)      0.6784
          Random Forest      0.7482
    Tuned Random Forest      0.7680

Selected Model: Logistic Regression


In [17]:
# Import cross_val_predict
from sklearn.model_selection import cross_val_predict

# Generate out-of-fold probabilities using training data only
oof_probs = cross_val_predict(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

# Import required metrics
from sklearn.metrics import precision_score, recall_score, f1_score

# Compare two decision thresholds
threshold_results = []

for threshold in [0.5, 0.35]:

    # Convert probabilities into binary predictions
    oof_pred = (oof_probs >= threshold).astype(int)

    # Calculate metrics
    precision = precision_score(y_train, oof_pred)
    recall = recall_score(y_train, oof_pred)
    f1 = f1_score(y_train, oof_pred)

    # Count appointments flagged as high risk
    flagged = oof_pred.sum()

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Flagged Appointments": flagged
    })

# Display comparison
threshold_df = pd.DataFrame(threshold_results)

print("Threshold Comparison (Training OOF Predictions)")
print("------------------------------------------------")
print(threshold_df.round(4).to_string(index=False))

Threshold Comparison (Training OOF Predictions)
------------------------------------------------
 Threshold  Precision  Recall  F1 Score  Flagged Appointments
      0.50     0.5753  0.2485    0.3471                    73
      0.35     0.5130  0.4675    0.4892                   154


**Threshold Selection observation**

A threshold of 0.35 was selected because it achieved a higher F1-score (0.4892) compared with the 0.50 threshold (0.3471). It also improved recall from 0.2485 to 0.4675, identifying more potential no-show appointments.

For a clinic, recall is important because a missed warning can result in an empty appointment slot, whereas making an additional follow-up call costs approximately one minute. The final threshold should also consider the front desk's calling capacity.

In [18]:
# Set the selected decision threshold (final model will use this threshold)
selected_threshold = 0.35

# Fit the final Logistic Regression Pipeline
lr_pipe.fit(X_train, y_train)

# Generate no-show probabilities on test data
final_probs = lr_pipe.predict_proba(X_test)[:, 1]

# Apply the selected threshold
final_pred = (final_probs >= selected_threshold).astype(int)

# Calculate final test metrics
final_precision = precision_score(y_test, final_pred)
final_recall = recall_score(y_test, final_pred)
final_f1 = f1_score(y_test, final_pred)
final_roc_auc = roc_auc_score(y_test, final_probs)

# Count flagged appointments
final_flagged = final_pred.sum()

# Display final results
print("Final Model: Logistic Regression")
print("---------------------------------")
print("Selected Threshold:", selected_threshold)
print("Precision:", round(final_precision, 4))
print("Recall:", round(final_recall, 4))
print("F1 Score:", round(final_f1, 4))
print("ROC-AUC:", round(final_roc_auc, 4))
print("Flagged Appointments:", final_flagged)

Final Model: Logistic Regression
---------------------------------
Selected Threshold: 0.35
Precision: 0.5714
Recall: 0.5714
F1 Score: 0.5714
ROC-AUC: 0.7891
Flagged Appointments: 49


In [19]:
# Create final risk scores DataFrame (export risk scores for test set)
risk_scores_df = pd.DataFrame({
    "appointment_id": df_ml.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": final_probs,
    "flag": final_pred
})

# Save risk scores to CSV
risk_scores_df.to_csv("test_risk_scores.csv", index=False)

# Verify exported data
print("Risk Scores Exported Successfully!")
print("Total Test Records:", len(risk_scores_df))
print("Flagged Appointments:", risk_scores_df["flag"].sum())

print("\nFirst 5 Risk Scores:")
print(risk_scores_df.head())

Risk Scores Exported Successfully!
Total Test Records: 198
Flagged Appointments: 49

First 5 Risk Scores:
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021569     0
1          A0336        0   0.150905     0
2          A0508        0   0.518698     1
3          A0314        0   0.191317     0
4          A0089        0   0.167730     0
